# Mission 3 최종 학습 — klue/roberta-large + soft-F1 (M3-ROBERTA-003 설정, Training 100%)

**팀 eigensinn.** 이 노트북은 Mission 3 제출 모델의 학습 로그입니다.

- **설정:** M3-ROBERTA-003과 같습니다. 80:20 홀드아웃에서 3 seed 평균 Macro F1 0.6427, 공식 Val(seed 42, 80% 학습) 0.6433이었습니다.
  - 모델: `klue/roberta-large`(revision `28d9112`), 앞 256토큰, [CLS] + dropout 0.1 + Linear(9)
  - 손실: BCE(pos_weight = clip(0.5·√(neg/pos), 1, 6)) + 1.0 × (1 − soft macro-F1)
  - 최적화: AdamW lr 1e-5, weight decay 0.01, linear warmup 10%, 실효 batch 16(8 × 2), fp16, 3 epochs, 마지막 epoch 사용
- **달라진 점:** Training 라벨 JSON **전체**로 학습합니다(내부 홀드아웃 없음). pos_weight도 전체 Training으로 계산합니다.
- **규정:**
  - 입력은 `utterances[].text`만 씁니다.
  - 판정은 모든 증상에 `확률 ≥ 0.5`이고 후처리는 없습니다.
  - 공식 Validation은 **마지막 채점에만** 씁니다. 학습, 가중치 계산, 선택에는 쓰지 않습니다.
- **실행:**
  - Jupyter가 있으면 위에서부터 셀을 실행합니다.
  - 없으면 터미널에서 실행합니다: `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 scripts/m3_final/m3_final_train.ipynb`


## 1. 경로와 seed 설정 (여기만 고치면 됩니다)

In [ ]:
from pathlib import Path
import os
REPO = Path.cwd()
while not (REPO / "src" / "dcc_inference").exists() and REPO != REPO.parent:
    REPO = REPO.parent          # works whether the kernel starts in the repo root or in scripts/m3_final
assert (REPO / "src" / "dcc_inference").exists(), "저장소 폴더를 찾지 못했습니다. 노트북을 클론한 저장소 안(scripts/m3_final)에서 실행하세요"
TRAIN_DIR   = Path(os.environ.get("M3_TRAIN_DIR", r"C:\dcc_data\Training\TL_서울_구급"))   # Training 라벨 JSON 폴더
VAL_DIR     = Path(os.environ.get("M3_VAL_DIR",   r"C:\dcc_data\Validation\VL_서울_구급"))  # 공식 Validation 라벨 JSON 폴더 (채점 전용)
OUT         = Path(os.environ.get("M3_OUT",       str(REPO / "runs" / "m3_final")))
MODEL_CACHE = Path(os.environ.get("M3_MODEL_CACHE", str(REPO / "model_cache")))
SEEDS       = [int(s) for s in os.environ.get("M3_SEEDS", "42 43 44").split()]
RUN_QUICK   = os.environ.get("M3_RUN_QUICK", "1") == "1"   # 1: 먼저 512행 1 epoch로 속도 확인
print("REPO", REPO); print("TRAIN_DIR", TRAIN_DIR, TRAIN_DIR.exists()); print("VAL_DIR", VAL_DIR, VAL_DIR.exists())
print("OUT", OUT); print("SEEDS", SEEDS, "RUN_QUICK", RUN_QUICK)
assert TRAIN_DIR.exists() and any(TRAIN_DIR.rglob("*.json")), "TRAIN_DIR에 라벨 JSON이 없습니다"


## 2. 실행 환경 확인 (계산 효율 보고용)

In [ ]:
import platform, subprocess, sys, torch, transformers
print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "| CUDA build", torch.version.cuda, "| transformers", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0),
          "| memory %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))
    torch.zeros(1, device="cuda").add_(1)   # RTX 50 시리즈에서 cu128 미만 빌드면 여기서 바로 실패합니다
    try:
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                             capture_output=True, text=True).stdout)
    except FileNotFoundError:
        pass
else:
    print("경고: GPU가 없습니다. 최종 학습은 GPU에서 하세요.")
print("training label JSONs:", sum(1 for _ in TRAIN_DIR.rglob("*.json")))


## 3. 속도 확인 (512행, 1 epoch — 최종 모델 아님)

In [ ]:
if RUN_QUICK:
    %run "{REPO}/scripts/m3_final/train_final.py" --train_dir "{TRAIN_DIR}" --out "{OUT}_quick" --seeds 42 --quick --model_cache "{MODEL_CACHE}"
else:
    print("skip")


## 4. 최종 학습 (Training 100%, seed별 3 epochs → `.pt` 내보내기 → 공식 Val 채점)

In [ ]:
seeds = " ".join(map(str, SEEDS))
%run "{REPO}/scripts/m3_final/train_final.py" --train_dir "{TRAIN_DIR}" --out "{OUT}" --seeds {seeds} --val_dir "{VAL_DIR}" --model_cache "{MODEL_CACHE}"


## 5. 결과 요약 (seed별 공식 Val, 그리고 seed 확률 평균)

In [ ]:
import json, numpy as np, unicodedata
from sklearn.metrics import f1_score
LABELS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
rows = []
for s in SEEDS:
    d = OUT / f"seed_{s}"
    hist = json.loads((d / "history.json").read_text(encoding="utf-8"))
    val = json.loads((d / "val_score.json").read_text(encoding="utf-8"))
    rows.append((s, val["macro_f1"], val["micro_f1"], val["empty_predictions"], sum(h["epoch_seconds"] for h in hist), val["inference_seconds"]))
    print(f"seed {s}: official Val Macro F1 {val['macro_f1']:.4f}  Micro {val['micro_f1']:.4f}  empty {val['empty_predictions']}  "
          f"train {sum(h['epoch_seconds'] for h in hist)/60:.1f} min  infer {val['inference_seconds']:.0f} s")
if len(SEEDS) > 1:
    # 사전에 정한 규칙: seed 확률의 단순 평균(가중치 선택 없음). 공식 Val은 평가에만 사용.
    paths = sorted(VAL_DIR.rglob("*.json"))
    gold = []
    for p in paths:
        sy = json.loads(p.read_text(encoding="utf-8-sig")).get("symptom", [])
        sy = {unicodedata.normalize("NFC", x.strip()) for x in ([sy] if isinstance(sy, str) else sy) if isinstance(x, str)}
        gold.append([int(l in sy) for l in LABELS])
    mean = np.mean([np.load(OUT / f"seed_{s}" / "val_probs.npy") for s in SEEDS], axis=0)
    pred = (mean >= 0.5).astype(int)
    print(f"seed average ({len(SEEDS)}): official Val Macro F1 {f1_score(np.array(gold), pred, average='macro', zero_division=0):.4f}")
for f in sorted(OUT.glob("*.pt")):
    print(f.name, f"{f.stat().st_size:,} bytes")


## 6. 마무리 (도서관 PC)
1. `runs/m3_final/*.pt`, `*.pt.json`, `seed_*/history.json`, `seed_*/val_score.json`, 그리고 **실행이 끝난 이 노트북**을 Google Drive나 USB로 옮깁니다.
2. 대회 데이터(`TRAIN_DIR`, `VAL_DIR`), `model_cache`, `runs`를 **도서관 PC에서 지웁니다.**
